# LeetCode 3354 — Make Array Elements Equal to Zero bằng Python

Link đề: https://leetcode.com/problems/make-array-elements-equal-to-zero/ · Độ khó: **Easy** · Chủ đề: Array, Prefix Sum (tổng tiền tố), Simulation (mô phỏng)

Skeleton LeetCode:

```python
class Solution:
    def countValidSelections(self, nums: List[int]) -> int:
        ...
```

Ghi chú: mọi đáp án trong notebook đã được kiểm chứng bằng code (đối chiếu công thức tổng tiền tố với **mô phỏng thật** trên mọi mảng nhỏ và hàng nghìn mảng ngẫu nhiên có seed). Hai ví dụ chính lấy từ đề; những ví dụ tự dựng thêm có ghi rõ "tự dựng" — nếu muốn khớp từng chữ thì nên đối chiếu lại đề gốc. Khi chạy ngoài LeetCode ta thêm `from typing import List` (LeetCode đã nhập sẵn).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: chọn một vị trí xuất phát `curr` có `nums[curr] == 0` và một hướng đi (trái/phải); rồi đi theo luật — gặp `0` thì **đi tiếp cùng hướng**, gặp số dương thì **giảm nó đi `1`, đảo hướng, và bước một bước** — cho tới khi ra ngoài mảng. Một lựa chọn (vị trí, hướng) là **hợp lệ** nếu cuối cùng **mọi phần tử đều bằng `0`**; đếm số lựa chọn hợp lệ.
2. Nắm các công cụ Python: `while` với điều kiện biên `0 <= curr < n`, biến hướng `step = ±1` và **đảo hướng bằng `step = -step`**, sao chép mảng bằng `list(nums)`, `sum`, tổng tiền tố chạy (running sum), `abs`, `all(...)`.
3. Đi từ cách **mô phỏng thật cho từng (vị trí, hướng)** → **công thức với tổng trái / tổng phải** (tính bằng cắt lát) → **tổng tiền tố chạy** (`O(n)`, bản chuẩn) → **mảng tiền tố dựng sẵn** → **xét riêng từng hướng bằng hàm điều kiện**.
4. Hiểu mấu chốt: người đi **luân phiên** "va" vào phần bên trái rồi phần bên phải (hoặc ngược lại), mỗi lần đụng làm giảm tổng của phía đó `1`. Hợp lệ khi hai tổng `L` (bên trái) và `R` (bên phải) của vị trí xuất phát khớp nhau: `L == R` (cả hai hướng tốt), hoặc chênh đúng `1` (chỉ đi về phía có tổng **lớn hơn** trước).
5. Tránh các lỗi kinh điển: mô phỏng trên chính `nums` (làm hỏng lần thử thứ hai), đếm `1` thay vì `2` khi `L == R`, chọn nhầm hướng khi chênh `1`, và quên chỉ xét các vị trí có giá trị `0`.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums`. Bắt đầu bằng cách chọn vị trí `curr` sao cho `nums[curr] == 0` và chọn **hướng** di chuyển là **trái** hoặc **phải**. Sau đó lặp lại quy trình:

- Nếu `curr` **ra ngoài** đoạn `[0, n - 1]` thì quy trình kết thúc.
- Nếu `nums[curr] == 0`: di chuyển **theo hướng hiện tại** (tăng `curr` nếu đang đi phải, giảm `curr` nếu đang đi trái).
- Ngược lại nếu `nums[curr] > 0`:
  - giảm `nums[curr]` đi `1`;
  - **đảo hướng** di chuyển (trái thành phải và ngược lại);
  - bước một bước theo hướng mới.

Một lựa chọn (vị trí ban đầu + hướng) là **hợp lệ** nếu **mọi phần tử của `nums` đều bằng `0`** khi quy trình kết thúc. Trả về **số lựa chọn hợp lệ**.

**Ví dụ 1**

- Input: `nums = [1,0,2,0,3]`
- Output: `2`
- Giải thích: hai lựa chọn hợp lệ duy nhất là chọn `curr = 3` và đi **trái**, hoặc chọn `curr = 3` và đi **phải**. (Đề liệt kê từng bước biến đổi mảng; ta sẽ tự trace ở mục 8.)

**Ví dụ 2**

- Input: `nums = [2,3,4,0,4,1,0]`
- Output: `0`
- Giải thích: không có lựa chọn hợp lệ.

**Ràng buộc**

- `1 <= nums.length <= 100`
- `0 <= nums[i] <= 100`
- Có ít nhất một phần tử `nums[i] == 0`.

In [1]:
from typing import List
import random

def simulate_once(nums, start, step):
    """Đi theo đúng luật đề. step = -1: đi trái, step = +1: đi phải. Trả True nếu mọi phần tử về 0."""
    arr = list(nums)               # copy để không làm hỏng input
    n = len(arr)
    curr = start
    while 0 <= curr < n:
        if arr[curr] == 0:
            curr += step
        else:
            arr[curr] -= 1
            step = -step
            curr += step
    return all(v == 0 for v in arr)

examples = [
    ([1, 0, 2, 0, 3], 2),
    ([2, 3, 4, 0, 4, 1, 0], 0),
]
for nums, expected in examples:
    count = sum(
        simulate_once(nums, i, step)
        for i, v in enumerate(nums) if v == 0
        for step in (-1, +1)
    )
    print(nums, "->", count, "(kỳ vọng", expected, ")")
    assert count == expected

[1, 0, 2, 0, 3] -> 2 (kỳ vọng 2 )
[2, 3, 4, 0, 4, 1, 0] -> 0 (kỳ vọng 0 )


## 3. Kiến thức nền

### 3.1. Hướng đi bằng `step = ±1` và đảo hướng bằng `step = -step`

Thay vì viết hai nhánh "đi trái" và "đi phải", ta lưu **hướng** trong một biến `step`: `-1` là sang trái (`curr` giảm `1`), `+1` là sang phải (`curr` tăng `1`). Mỗi bước: `curr += step`. **Đảo hướng**: `step = -step` (`-1` thành `+1` và ngược lại).

In [2]:
step = -1
curr = 3
curr += step
print(curr)          # 2  (đi trái)
step = -step         # đảo hướng
print(step)          # 1
curr += step
print(curr)          # 3  (đi phải)

2
1
3


### 3.2. Điều kiện biên `0 <= curr < n` trong `while`

Quy trình kết thúc khi `curr` **ra ngoài** mảng. Vòng `while 0 <= curr < n:` (Python cho phép so sánh liên hoàn) chạy **chừng nào còn trong mảng**. Ra khỏi vòng lặp tức là `curr < 0` hoặc `curr >= n`. Lưu ý Python cho phép chỉ số âm (`arr[-1]` là phần tử cuối) nên **phải kiểm tra biên trước** khi đọc `arr[curr]`.

In [3]:
n = 5
for curr in (-1, 0, 4, 5):
    print(curr, "còn trong mảng?", 0 <= curr < n)

-1 còn trong mảng? False
0 còn trong mảng? True
4 còn trong mảng? True
5 còn trong mảng? False


### 3.3. Sao chép mảng: `list(nums)` hay `nums[:]` (đừng gán thẳng)

`b = a` **không** tạo bản sao — `a` và `b` cùng trỏ **một** danh sách. Sửa `b` thì `a` đổi theo. Mỗi lần mô phỏng mảng bị sửa (`arr[curr] -= 1`), nên phải **làm việc trên bản sao**.

In [4]:
a = [1, 0, 2]
b = a                 # cùng một danh sách
b[0] = 99
print(a)              # [99, 0, 2]  <- a bị đổi!

a = [1, 0, 2]
c = list(a)           # bản sao thật
c[0] = 99
print(a, c)           # [1, 0, 2] [99, 0, 2]

[99, 0, 2]
[1, 0, 2] [99, 0, 2]


### 3.4. Tổng tiền tố chạy (running sum)

Để biết "tổng bên trái" `L` của vị trí `i` ta cộng dồn khi duyệt: trước khi xét `nums[i]`, biến `left` là tổng các phần tử `nums[0..i-1]`. Sau khi xét xong ta cộng `nums[i]` vào `left`. Tổng bên phải: `R = total - L - nums[i]`; với `nums[i] = 0` thì `R = total - L`.

In [5]:
nums = [1, 0, 2, 0, 3]
total = sum(nums)
left = 0
for i, x in enumerate(nums):
    print(f"i={i}, x={x}, tổng trái L={left}, tổng phải R={total - left - x}")
    left += x

i=0, x=1, tổng trái L=0, tổng phải R=5
i=1, x=0, tổng trái L=1, tổng phải R=5
i=2, x=2, tổng trái L=1, tổng phải R=3
i=3, x=0, tổng trái L=3, tổng phải R=3
i=4, x=3, tổng trái L=3, tổng phải R=0


### 3.5. `abs`, `all`, và so sánh với chênh `1`

- `abs(L - R)` là **độ chênh** giữa hai tổng, không phụ thuộc dấu.
- `all(v == 0 for v in arr)` trả `True` nếu **mọi** phần tử bằng `0` (đúng điều kiện "hợp lệ").

In [6]:
print(abs(3 - 5), abs(5 - 3))                 # 2 2
print(all(v == 0 for v in [0, 0, 0]))         # True
print(all(v == 0 for v in [0, 1, 0]))         # False
print(all(v == 0 for v in []))                # True (rỗng thì tầm thường)

2 2
True
False
True


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Hãy theo dõi một lựa chọn trên `nums = [1,0,2,0,3]`: bắt đầu `curr = 3` (giá trị `0`), đi **trái**.

- Vị trí 3 là `0` → đi trái tới vị trí 2.
- Vị trí 2 là `2` → giảm còn `1`, đảo hướng thành **phải**, bước tới vị trí 3.
- Vị trí 3 là `0` → đi phải tới vị trí 4.
- Vị trí 4 là `3` → giảm còn `2`, đảo thành **trái**, bước tới 3 → ...

Ta thấy người đi **như quả bóng nảy**: bên trái có tổng `L = 1 + 0 + 2 = 3`, bên phải có tổng `R = 3`. Mỗi lần va vào một phía thì tổng phía đó giảm `1`; và các va chạm **luân phiên trái, phải, trái, phải...**. Nếu `L = R`: sau `L` cặp va chạm cả hai phía cùng về `0` và người đi thoát ra ngoài. Nếu `L` và `R` lệch nhau thì phía "dư" vẫn còn số dương khi người đi đã thoát ra (hoặc không có đủ va chạm) — thất bại.

Nhìn nhanh cho `[2,3,4,0,4,1,0]`: tại vị trí `3`: `L = 9`, `R = 5` (chênh `4`) → không hợp lệ. Tại vị trí `6`: `L = 14`, `R = 0` → không hợp lệ.

In [7]:
for nums in ([1, 0, 2, 0, 3], [2, 3, 4, 0, 4, 1, 0]):
    total = sum(nums)
    print("nums =", nums, "tổng =", total)
    left = 0
    for i, x in enumerate(nums):
        if x == 0:
            print(f"  vị trí {i}: L={left}, R={total - left}, |L-R|={abs(left - (total - left))}")
        left += x

nums = [1, 0, 2, 0, 3] tổng = 6
  vị trí 1: L=1, R=5, |L-R|=4
  vị trí 3: L=3, R=3, |L-R|=0
nums = [2, 3, 4, 0, 4, 1, 0] tổng = 14
  vị trí 3: L=9, R=5, |L-R|=4
  vị trí 6: L=14, R=0, |L-R|=14


## 5. Cách 1 — Brute Force / baseline: mô phỏng thật cho từng (vị trí, hướng)

**Ý tưởng.** Làm đúng như đề: với mọi `i` có `nums[i] == 0` và mọi hướng (`-1`, `+1`), **chạy mô phỏng** trên một bản sao, rồi kiểm tra mọi phần tử đã bằng `0`. Đếm số lần hợp lệ.

Hàm `simulate_once` ở mục 2 chính là mô phỏng; ta chỉ việc đếm.

In [8]:
def count_valid_simulate(nums: List[int]) -> int:
    count = 0
    for i, v in enumerate(nums):
        if v == 0:
            for step in (-1, +1):
                if simulate_once(nums, i, step):
                    count += 1
    return count

print(count_valid_simulate([1, 0, 2, 0, 3]))             # 2
print(count_valid_simulate([2, 3, 4, 0, 4, 1, 0]))       # 0
print(count_valid_simulate([0]))                         # 2 (đi trái hoặc phải đều ra ngay, mảng đã toàn 0)

2
0
2


**Độ phức tạp (Cách 1).** Gọi `n = len(nums)` và `S = sum(nums)`. Mỗi lần mô phỏng, mỗi bước hoặc **đi qua một số `0`**, hoặc **giảm một số dương** (làm `S` giảm `1`); giữa hai lần giảm, ta đi qua tối đa `n` số `0`. Vậy một lần mô phỏng tốn `O(n · S + n)`; có tới `2n` lựa chọn: tổng **`O(n² · S)`** — với `n = 100`, `S` tới `10^4` đó là `10^10` thao tác **trong trường hợp xấu nhất lý thuyết** (thực tế nhiều lựa chọn thất bại rất sớm). Space `O(n)` cho bản sao.

**Nút thắt.** Ta **không cần chạy từng bước** để biết kết quả: chỉ cần biết **tổng bên trái và tổng bên phải** của vị trí xuất phát.

## 6. Ý tưởng thuật toán / bước đột phá

Xuất phát tại một vị trí có giá trị `0`. Gọi `L` = tổng các phần tử **bên trái**, `R` = tổng các phần tử **bên phải**. Người đi **luân phiên** va vào phía trái rồi phải (hoặc phải rồi trái, tuỳ hướng ban đầu); mỗi va chạm giảm `1` đơn vị của phía bị va. Quy trình chấm dứt khi ra khỏi mảng, và chỉ ra được khi **một phía đã hết số dương theo hướng đang đi**.

Đi hướng **trái trước**: chuỗi va chạm là `T, P, T, P, ...` (T = trái, P = phải).

- Hợp lệ khi **`L = R`**: va chạm cuối cùng là `P` (lần thứ `R`), sau đó người đi quay sang trái và thoát; cả hai phía đều về `0`.
- Hợp lệ khi **`L = R + 1`**: va chạm cuối cùng là `T` (lần thứ `L`), sau đó người đi quay sang phải và thoát; cả hai phía đều về `0`.
- Mọi trường hợp khác thì một phía còn dư số dương lúc thoát (hoặc người đi thoát từ phía chưa hết) ⇒ **không hợp lệ**.

Đi **phải trước** thì đối xứng: hợp lệ khi `R = L` hoặc `R = L + 1`.

**Kết luận:** với mỗi vị trí `i` có `nums[i] == 0`:

- `L == R` ⇒ cộng **`2`** (cả hai hướng);
- `L - R == 1` ⇒ cộng `1` (đi **trái**);
- `R - L == 1` ⇒ cộng `1` (đi **phải**);
- còn lại ⇒ cộng `0`.

In [9]:
def valid_directions(left_sum, right_sum):
    """Trả về (đi_trái_hợp_lệ, đi_phải_hợp_lệ) theo công thức."""
    go_left = left_sum == right_sum or left_sum == right_sum + 1
    go_right = left_sum == right_sum or right_sum == left_sum + 1
    return go_left, go_right

for L, R in [(3, 3), (4, 3), (3, 4), (3, 5), (0, 0), (1, 0), (0, 1), (9, 5)]:
    print(f"L={L}, R={R} -> (trái, phải) = {valid_directions(L, R)}")

L=3, R=3 -> (trái, phải) = (True, True)
L=4, R=3 -> (trái, phải) = (True, False)
L=3, R=4 -> (trái, phải) = (False, True)
L=3, R=5 -> (trái, phải) = (False, False)
L=0, R=0 -> (trái, phải) = (True, True)
L=1, R=0 -> (trái, phải) = (True, False)
L=0, R=1 -> (trái, phải) = (False, True)
L=9, R=5 -> (trái, phải) = (False, False)


## 7. Các cách tiếp theo

### Cách 2 — Công thức với tổng trái / tổng phải bằng cắt lát

**Giải thích từng đoạn.**

1. Với mỗi `i` mà `nums[i] == 0`: `L = sum(nums[:i])`, `R = sum(nums[i + 1:])`.
2. Áp dụng quy tắc ở mục 6: `L == R` → `+2`; `abs(L - R) == 1` → `+1` (đúng một hướng).
3. Cộng dồn vào `ans`.

Cách này **dễ hiểu nhất** nhưng tính lại tổng cho mỗi `i` nên `O(n²)`.

In [10]:
def count_valid_slice(nums: List[int]) -> int:
    ans = 0
    for i, x in enumerate(nums):
        if x != 0:
            continue
        L = sum(nums[:i])
        R = sum(nums[i + 1:])
        if L == R:
            ans += 2
        elif abs(L - R) == 1:
            ans += 1
    return ans

print(count_valid_slice([1, 0, 2, 0, 3]))             # 2
print(count_valid_slice([2, 3, 4, 0, 4, 1, 0]))       # 0
print(count_valid_slice([1, 0]))                      # 1

2
0
1


In [11]:
# Code nộp LeetCode cho Cách 2 (đổi tên lớp thành Solution khi nộp)
class SolutionSlice:
    def countValidSelections(self, nums: List[int]) -> int:
        ans = 0
        for i, x in enumerate(nums):
            if x != 0:
                continue
            L = sum(nums[:i])
            R = sum(nums[i + 1:])
            if L == R:
                ans += 2
            elif abs(L - R) == 1:
                ans += 1
        return ans

print(SolutionSlice().countValidSelections([1, 0, 2, 0, 3]))   # 2

2


**Độ phức tạp Cách 2:** **Time `O(n²)`** (mỗi `i` tính hai tổng `O(n)`), **Space `O(n)`** (cắt lát tạo bản sao tạm).

### Cách 3 — Tổng tiền tố chạy: một lượt `O(n)` (lời giải chính thức)

**Giải thích từng đoạn.**

1. `s = sum(nums)` — tổng toàn mảng, tính một lần.
2. `l = 0` — tổng bên trái của vị trí hiện tại.
3. Duyệt `x` trong `nums`: nếu `x != 0` thì `l += x` (đưa nó sang "bên trái" của các vị trí sau). Nếu `x == 0`, thì tổng phải là `s - l` (vì `x = 0`). So `l * 2` với `s`: `l == s - l` ⇔ `2l == s`; `|l - (s - l)| = |2l - s|`.
4. `2l == s` → `+2`; `|2l - s| == 1` → `+1`.

In [12]:
def count_valid_prefix_running(nums: List[int]) -> int:
    s = sum(nums)
    ans = l = 0
    for x in nums:
        if x:
            l += x
        elif l * 2 == s:
            ans += 2
        elif abs(l * 2 - s) == 1:
            ans += 1
    return ans

print(count_valid_prefix_running([1, 0, 2, 0, 3]))             # 2
print(count_valid_prefix_running([2, 3, 4, 0, 4, 1, 0]))       # 0
print(count_valid_prefix_running([0, 0]))                      # 4

2
0
4


In [13]:
# Code nộp LeetCode cho Cách 3
class SolutionRunning:
    def countValidSelections(self, nums: List[int]) -> int:
        s = sum(nums)
        ans = l = 0
        for x in nums:
            if x:
                l += x
            elif l * 2 == s:
                ans += 2
            elif abs(l * 2 - s) == 1:
                ans += 1
        return ans

print(SolutionRunning().countValidSelections([0, 0]))   # 4

4


**Độ phức tạp Cách 3:** **Time `O(n)`**, **Space `O(1)`**.

### Cách 4 — Mảng tiền tố dựng sẵn

**Giải thích.** Dựng `pref[i]` = tổng `nums[0..i-1]` (`pref[0] = 0`, `pref[n] = tổng`). Khi đó tổng trái của `i` là `pref[i]`, tổng phải là `pref[n] - pref[i + 1]`. Cách này giúp tính tổng của **mọi đoạn** trong `O(1)` và phù hợp khi phải trả lời nhiều truy vấn trên cùng một mảng.

In [14]:
def count_valid_prefix_array(nums: List[int]) -> int:
    n = len(nums)
    pref = [0] * (n + 1)
    for i, x in enumerate(nums):
        pref[i + 1] = pref[i] + x
    ans = 0
    for i, x in enumerate(nums):
        if x != 0:
            continue
        L = pref[i]
        R = pref[n] - pref[i + 1]
        if L == R:
            ans += 2
        elif abs(L - R) == 1:
            ans += 1
    return ans

nums = [1, 0, 2, 0, 3]
pref = [0]
for x in nums:
    pref.append(pref[-1] + x)
print(pref)                                                   # [0, 1, 1, 3, 3, 6]
print(count_valid_prefix_array([1, 0, 2, 0, 3]))              # 2
print(count_valid_prefix_array([2, 3, 4, 0, 4, 1, 0]))        # 0

[0, 1, 1, 3, 3, 6]
2
0


In [15]:
# Code nộp LeetCode cho Cách 4
class SolutionPrefixArray:
    def countValidSelections(self, nums: List[int]) -> int:
        n = len(nums)
        pref = [0] * (n + 1)
        for i, x in enumerate(nums):
            pref[i + 1] = pref[i] + x
        ans = 0
        for i, x in enumerate(nums):
            if x != 0:
                continue
            L = pref[i]
            R = pref[n] - pref[i + 1]
            if L == R:
                ans += 2
            elif abs(L - R) == 1:
                ans += 1
        return ans

print(SolutionPrefixArray().countValidSelections([1, 0, 2, 0, 3]))   # 2

2


**Độ phức tạp Cách 4:** **Time `O(n)`**, **Space `O(n)`** (mảng `pref`).

### Cách 5 — Xét riêng từng hướng bằng hàm điều kiện

**Giải thích.** Tách thành hai câu hỏi độc lập: "đi **trái** từ `i` có hợp lệ không?" và "đi **phải** từ `i` có hợp lệ không?". Với trái: `L == R` hoặc `L == R + 1`. Với phải: `R == L` hoặc `R == L + 1`. Cách viết này **bám đúng chứng minh** (mục 6) và dễ mở rộng khi đề đổi luật.

In [16]:
def can_go_left(L, R):
    return L - R in (0, 1)        # L == R hoặc L == R + 1

def can_go_right(L, R):
    return R - L in (0, 1)        # R == L hoặc R == L + 1

def count_valid_by_direction(nums: List[int]) -> int:
    total = sum(nums)
    left = 0
    ans = 0
    for x in nums:
        if x == 0:
            right = total - left
            ans += can_go_left(left, right) + can_go_right(left, right)   # True + True = 2
        left += x
    return ans

print(count_valid_by_direction([1, 0, 2, 0, 3]))             # 2
print(count_valid_by_direction([2, 3, 4, 0, 4, 1, 0]))       # 0
print(can_go_left(4, 3), can_go_right(4, 3))                 # True False

2
0
True False


In [17]:
# Code nộp LeetCode cho Cách 5
class SolutionDirection:
    def countValidSelections(self, nums: List[int]) -> int:
        total = sum(nums)
        left = 0
        ans = 0
        for x in nums:
            if x == 0:
                right = total - left
                if left - right in (0, 1):      # đi trái
                    ans += 1
                if right - left in (0, 1):      # đi phải
                    ans += 1
            left += x
        return ans

print(SolutionDirection().countValidSelections([1, 0, 2, 0, 3]))   # 2

2


**Độ phức tạp Cách 5:** **Time `O(n)`**, **Space `O(1)`**.

## 8. TRACE từng bước

Hai trace: (a) **mô phỏng thật** `[1,0,2,0,3]`, bắt đầu `curr = 3`, hướng **trái** — để thấy người đi "nảy" qua lại; (b) **vòng lặp tiền tố** (Cách 3) trên `[1,0,2,0,3]`.

In [18]:
def trace_simulate(nums, start, step):
    arr = list(nums)
    n = len(arr)
    curr = start
    name = {-1: "trái", 1: "phải"}
    print(f"bắt đầu: curr={curr}, hướng={name[step]}, mảng={arr}")
    t = 0
    while 0 <= curr < n:
        t += 1
        if arr[curr] == 0:
            print(f"  bước {t:>2}: curr={curr}, giá trị 0 -> đi tiếp {name[step]}")
            curr += step
        else:
            arr[curr] -= 1
            step = -step
            print(f"  bước {t:>2}: curr={curr}, giá trị dương -> giảm còn {arr[curr]}, đảo sang {name[step]}, mảng={arr}")
            curr += step
    ok = all(v == 0 for v in arr)
    print("kết thúc: curr =", curr, "(ra ngoài), mảng =", arr, "-> hợp lệ?", ok)
    return ok

trace_simulate([1, 0, 2, 0, 3], 3, -1)

bắt đầu: curr=3, hướng=trái, mảng=[1, 0, 2, 0, 3]
  bước  1: curr=3, giá trị 0 -> đi tiếp trái
  bước  2: curr=2, giá trị dương -> giảm còn 1, đảo sang phải, mảng=[1, 0, 1, 0, 3]
  bước  3: curr=3, giá trị 0 -> đi tiếp phải
  bước  4: curr=4, giá trị dương -> giảm còn 2, đảo sang trái, mảng=[1, 0, 1, 0, 2]
  bước  5: curr=3, giá trị 0 -> đi tiếp trái
  bước  6: curr=2, giá trị dương -> giảm còn 0, đảo sang phải, mảng=[1, 0, 0, 0, 2]
  bước  7: curr=3, giá trị 0 -> đi tiếp phải
  bước  8: curr=4, giá trị dương -> giảm còn 1, đảo sang trái, mảng=[1, 0, 0, 0, 1]
  bước  9: curr=3, giá trị 0 -> đi tiếp trái
  bước 10: curr=2, giá trị 0 -> đi tiếp trái
  bước 11: curr=1, giá trị 0 -> đi tiếp trái
  bước 12: curr=0, giá trị dương -> giảm còn 0, đảo sang phải, mảng=[0, 0, 0, 0, 1]
  bước 13: curr=1, giá trị 0 -> đi tiếp phải
  bước 14: curr=2, giá trị 0 -> đi tiếp phải
  bước 15: curr=3, giá trị 0 -> đi tiếp phải
  bước 16: curr=4, giá trị dương -> giảm còn 0, đảo sang trái, mảng=[0, 0, 0, 0,

True

In [19]:
trace_simulate([1, 0, 2, 0, 3], 1, -1)      # vị trí 1: L=1, R=5 -> thất bại
print()
trace_simulate([1, 0, 2, 0, 3], 3, +1)      # đi phải từ vị trí 3 cũng hợp lệ (L = R = 3)

bắt đầu: curr=1, hướng=trái, mảng=[1, 0, 2, 0, 3]
  bước  1: curr=1, giá trị 0 -> đi tiếp trái
  bước  2: curr=0, giá trị dương -> giảm còn 0, đảo sang phải, mảng=[0, 0, 2, 0, 3]
  bước  3: curr=1, giá trị 0 -> đi tiếp phải
  bước  4: curr=2, giá trị dương -> giảm còn 1, đảo sang trái, mảng=[0, 0, 1, 0, 3]
  bước  5: curr=1, giá trị 0 -> đi tiếp trái
  bước  6: curr=0, giá trị 0 -> đi tiếp trái
kết thúc: curr = -1 (ra ngoài), mảng = [0, 0, 1, 0, 3] -> hợp lệ? False

bắt đầu: curr=3, hướng=phải, mảng=[1, 0, 2, 0, 3]
  bước  1: curr=3, giá trị 0 -> đi tiếp phải
  bước  2: curr=4, giá trị dương -> giảm còn 2, đảo sang trái, mảng=[1, 0, 2, 0, 2]
  bước  3: curr=3, giá trị 0 -> đi tiếp trái
  bước  4: curr=2, giá trị dương -> giảm còn 1, đảo sang phải, mảng=[1, 0, 1, 0, 2]
  bước  5: curr=3, giá trị 0 -> đi tiếp phải
  bước  6: curr=4, giá trị dương -> giảm còn 1, đảo sang trái, mảng=[1, 0, 1, 0, 1]
  bước  7: curr=3, giá trị 0 -> đi tiếp trái
  bước  8: curr=2, giá trị dương -> giảm còn 0,

True

In [20]:
def trace_prefix(nums):
    s = sum(nums)
    print(f"nums={nums}, s={s}")
    ans = l = 0
    for i, x in enumerate(nums):
        if x:
            l += x
            print(f"i={i}, x={x}: số dương -> l={l}")
        else:
            r = s - l
            if l * 2 == s:
                ans += 2
                action = "L == R -> +2"
            elif abs(l * 2 - s) == 1:
                ans += 1
                action = "chênh 1 -> +1"
            else:
                action = "không hợp lệ -> +0"
            print(f"i={i}, x=0: L={l}, R={r}, {action}, ans={ans}")
    return ans

trace_prefix([1, 0, 2, 0, 3])
print()
trace_prefix([2, 3, 4, 0, 4, 1, 0])

nums=[1, 0, 2, 0, 3], s=6
i=0, x=1: số dương -> l=1
i=1, x=0: L=1, R=5, không hợp lệ -> +0, ans=0
i=2, x=2: số dương -> l=3
i=3, x=0: L=3, R=3, L == R -> +2, ans=2
i=4, x=3: số dương -> l=6

nums=[2, 3, 4, 0, 4, 1, 0], s=14
i=0, x=2: số dương -> l=2
i=1, x=3: số dương -> l=5
i=2, x=4: số dương -> l=9
i=3, x=0: L=9, R=5, không hợp lệ -> +0, ans=0
i=4, x=4: số dương -> l=13
i=5, x=1: số dương -> l=14
i=6, x=0: L=14, R=0, không hợp lệ -> +0, ans=0


0

Bảng trace Cách 3 với `nums = [1, 0, 2, 0, 3]` (`s = 6`):

| i | x | `l` trước | `R = s − l` | Hành động | `ans` |
|---|---|---|---|---|---|
| 0 | 1 | 0 | — | dương: `l = 1` | 0 |
| 1 | 0 | 1 | 5 | `2l = 2 ≠ 6`, `|2−6| = 4` → không | 0 |
| 2 | 2 | 1 | — | dương: `l = 3` | 0 |
| 3 | 0 | 3 | 3 | `2l = 6 = s` → `+2` | 2 |
| 4 | 3 | 3 | — | dương: `l = 6` | 2 |

Đáp án `2` — hai lựa chọn: `curr = 3` đi trái và `curr = 3` đi phải (đúng như đề).

## 9. Code nộp LeetCode

Bản nộp dùng Cách 3 (tổng tiền tố chạy), `O(n)` thời gian, `O(1)` bộ nhớ.

In [21]:
class Solution:
    def countValidSelections(self, nums: List[int]) -> int:
        s = sum(nums)
        ans = l = 0
        for x in nums:
            if x:
                l += x
            elif l * 2 == s:
                ans += 2
            elif abs(l * 2 - s) == 1:
                ans += 1
        return ans

sol = Solution()
print(sol.countValidSelections([1, 0, 2, 0, 3]))           # 2
print(sol.countValidSelections([2, 3, 4, 0, 4, 1, 0]))     # 0

2
0


## 10. Vì sao thuật toán đúng

Xét xuất phát tại vị trí `i` có `nums[i] = 0`; `L` là tổng bên trái, `R` là tổng bên phải.

- **Mỗi lần đụng một số dương** ta giảm nó đúng `1`, đảo hướng và đi tiếp, nên người đi **không bao giờ đụng liền hai lần cùng một phía** nếu ở phía kia vẫn còn số dương: sau khi đụng phía trái, hướng đổi sang phải, đi qua các số `0` cho tới số dương **đầu tiên bên phải** hoặc **ra ngoài** (nếu bên phải đã hết số dương). Vậy các lần đụng **luân phiên** trái-phải.
- **Điều kiện kết thúc đúng:** người đi chỉ thoát khi đi về một phía mà phía đó **không còn số dương**. Tất cả phần tử bằng `0` đúng khi **cả `L` lẫn `R`** bị "tiêu thụ" đủ: tức số lần đụng bên trái bằng `L` và bên phải bằng `R`.
- **Xuất phát đi trái:** chuỗi đụng là `T, P, T, P, ...`. Sau `a` lần đụng trái và `b` lần đụng phải ta có `a = b` hoặc `a = b + 1`. Cần `a = L`, `b = R` ⇒ `L = R` hoặc `L = R + 1`. Nếu `L = R`, đụng cuối là `P` rồi quay sang trái và thoát (bên trái đã hết). Nếu `L = R + 1`, đụng cuối là `T` rồi quay sang phải và thoát (bên phải đã hết). Nếu `L < R`, sau `L` lần đụng trái, đi trái sẽ thoát khi còn `R - L` đơn vị bên phải; nếu `L > R + 1`, bên phải hết trước và người đi thoát về bên phải khi bên trái còn dư.
- **Đối xứng cho hướng phải**, và khi `L = R` cả hai hướng đều tốt nên cộng `2`.
- Chỉ các vị trí `nums[i] == 0` được chọn làm xuất phát (đề yêu cầu), và đề đảm bảo có ít nhất một `0`.
- Bộ test ở mục 12 còn **đối chiếu với mô phỏng thật** trên mọi mảng nhỏ, nên công thức không chỉ "tin được" mà còn được kiểm chứng.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums)`, `S = sum(nums)`.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1 — mô phỏng thật | `O(n² · S)` (lý thuyết), chạy tốt với ràng buộc nhỏ | `O(n)` | đúng nhất, dùng để kiểm chứng |
| 2 — công thức + cắt lát | `O(n²)` | `O(n)` | dễ hiểu |
| 3 — tổng tiền tố chạy | `O(n)` | `O(1)` | chuẩn nộp bài |
| 4 — mảng tiền tố | `O(n)` | `O(n)` | nhiều truy vấn tổng đoạn |
| 5 — xét từng hướng | `O(n)` | `O(1)` | bám chứng minh |

## 12. Bộ test

Gom các cách vào `impls`; thêm `count_valid_simulate` (mô phỏng thật) làm **oracle**.

In [22]:
impls = {
    "simulate": count_valid_simulate,
    "slice": count_valid_slice,
    "running": count_valid_prefix_running,
    "prefix_array": count_valid_prefix_array,
    "direction": count_valid_by_direction,
    "Solution": Solution().countValidSelections,
    "SolutionSlice": SolutionSlice().countValidSelections,
    "SolutionRunning": SolutionRunning().countValidSelections,
    "SolutionPrefixArray": SolutionPrefixArray().countValidSelections,
    "SolutionDirection": SolutionDirection().countValidSelections,
}

def check_all(nums, expected, note=""):
    for name, f in impls.items():
        got = f(list(nums))
        assert got == expected, (name, nums, got, expected)
    print(f"OK  {nums} -> {expected}  {note}")

check_all([1, 0, 2, 0, 3], 2, "ví dụ 1")
check_all([2, 3, 4, 0, 4, 1, 0], 0, "ví dụ 2")

OK  [1, 0, 2, 0, 3] -> 2  ví dụ 1
OK  [2, 3, 4, 0, 4, 1, 0] -> 0  ví dụ 2


### 12.1. Các trường hợp biên

In [23]:
check_all([0], 2, "một phần tử 0: ra ngoài ngay, mảng đã toàn 0, cả hai hướng hợp lệ")
check_all([0, 0], 4, "hai số 0: mỗi vị trí 2 hướng")
check_all([0, 0, 0, 0], 8, "toàn 0: n vị trí × 2 hướng")
check_all([1, 0], 1, "chỉ đi trái hợp lệ (L = R + 1)")
check_all([0, 1], 1, "chỉ đi phải hợp lệ (R = L + 1)")
check_all([2, 0], 0, "L = 2, R = 0: chênh 2")
check_all([1, 0, 1], 2, "L = R = 1: cả hai hướng")
check_all([1, 1, 0, 1], 1, "L = 2, R = 1: chỉ đi trái (tự dựng)")
check_all([1, 0, 1, 1], 1, "L = 1, R = 2: chỉ đi phải (tự dựng)")
check_all([100, 0, 100], 2, "giá trị lớn nhất, L = R = 100")
check_all([5, 0, 0, 5], 4, "hai số 0 cùng có L = R = 5")
check_all([3, 0, 0, 0], 0, "L = 3, R = 0 ở mọi vị trí 0 -> không hợp lệ (tự dựng)")

OK  [0] -> 2  một phần tử 0: ra ngoài ngay, mảng đã toàn 0, cả hai hướng hợp lệ
OK  [0, 0] -> 4  hai số 0: mỗi vị trí 2 hướng
OK  [0, 0, 0, 0] -> 8  toàn 0: n vị trí × 2 hướng
OK  [1, 0] -> 1  chỉ đi trái hợp lệ (L = R + 1)
OK  [0, 1] -> 1  chỉ đi phải hợp lệ (R = L + 1)
OK  [2, 0] -> 0  L = 2, R = 0: chênh 2
OK  [1, 0, 1] -> 2  L = R = 1: cả hai hướng
OK  [1, 1, 0, 1] -> 1  L = 2, R = 1: chỉ đi trái (tự dựng)
OK  [1, 0, 1, 1] -> 1  L = 1, R = 2: chỉ đi phải (tự dựng)
OK  [100, 0, 100] -> 2  giá trị lớn nhất, L = R = 100
OK  [5, 0, 0, 5] -> 4  hai số 0 cùng có L = R = 5
OK  [3, 0, 0, 0] -> 0  L = 3, R = 0 ở mọi vị trí 0 -> không hợp lệ (tự dựng)


### 12.2. Phản ví dụ cho các cách làm sai

Ba cách sai hay gặp: (a) đếm `1` thay vì `2` khi `L == R`; (b) khi chênh `1` lại cộng `2` (nhầm là cả hai hướng đều hợp lệ); (c) không kiểm tra `x == 0` (xét cả vị trí có số dương). Ta tìm input làm chúng lệch so với oracle mô phỏng.

In [24]:
def wrong_equal_gives_1(nums):
    s = sum(nums); ans = l = 0
    for x in nums:
        if x: l += x
        elif l * 2 == s: ans += 1            # SAI: phải +2
        elif abs(l * 2 - s) == 1: ans += 1
    return ans

def wrong_diff1_gives_2(nums):
    s = sum(nums); ans = l = 0
    for x in nums:
        if x: l += x
        elif l * 2 == s: ans += 2
        elif abs(l * 2 - s) == 1: ans += 2   # SAI: chỉ một hướng hợp lệ
    return ans

def wrong_not_only_zero(nums):
    s = sum(nums); ans = 0; l = 0
    for x in nums:
        if l * 2 == s: ans += 2              # SAI: xét cả vị trí số dương
        elif abs(l * 2 - s) == 1: ans += 1
        l += x
    return ans

def find_counterexample(wrong, tries=20000, seed=5):
    rng = random.Random(seed)
    for _ in range(tries):
        n = rng.randint(1, 6)
        nums = [rng.randint(0, 3) for _ in range(n)]
        if 0 not in nums:
            nums[rng.randrange(n)] = 0
        if wrong(nums) != count_valid_simulate(nums):
            return nums, wrong(nums), count_valid_simulate(nums)

for w in (wrong_equal_gives_1, wrong_diff1_gives_2, wrong_not_only_zero):
    ce = find_counterexample(w)
    print(w.__name__, "-> (nums, sai, đúng) =", ce)
    assert ce is not None

wrong_equal_gives_1 -> (nums, sai, đúng) = ([2, 2, 0, 3, 1], 1, 2)
wrong_diff1_gives_2 -> (nums, sai, đúng) = ([0, 1], 2, 1)
wrong_not_only_zero -> (nums, sai, đúng) = ([2, 2, 0, 3, 1], 4, 2)


### 12.3. Đối chiếu với mô phỏng thật: vét cạn và ngẫu nhiên (có seed)

**(1)** Mọi mảng độ dài `1..6` với giá trị `0..3` **có ít nhất một số `0`** (đúng ràng buộc): so mọi cách với mô phỏng.
**(2)** Ngẫu nhiên có seed: `n <= 30`, giá trị `0..10` (để mô phỏng chạy nhanh) và có ít nhất một `0`.
**(3)** Ngẫu nhiên có seed **đúng toàn bộ ràng buộc** (`n <= 100`, giá trị `0..100`): chỉ so các cách `O(n)`/`O(n²)` với nhau (không chạy mô phỏng vì có thể rất chậm).

In [25]:
import itertools

total = 0
for n in range(1, 7):
    for tup in itertools.product(range(4), repeat=n):
        if 0 not in tup:
            continue
        nums = list(tup)
        expected = count_valid_simulate(nums)
        for name, f in impls.items():
            assert f(nums) == expected, (name, nums, expected)
        total += 1
print("vét cạn", total, "mảng nhỏ so với mô phỏng: OK")

rng = random.Random(3354)
nonzero_answers = 0
for trial in range(600):
    n = rng.randint(1, 30)
    nums = [rng.randint(0, 10) for _ in range(n)]
    nums[rng.randrange(n)] = 0
    expected = count_valid_simulate(nums)
    for name, f in impls.items():
        assert f(nums) == expected, (name, nums, expected)
    nonzero_answers += expected > 0
print("ngẫu nhiên nhỏ có seed: OK (số ca đáp án > 0:", nonzero_answers, ")")

fast = {k: v for k, v in impls.items() if k != "simulate"}
for trial in range(2000):
    n = rng.randint(1, 100)
    nums = [rng.randint(0, 100) for _ in range(n)]
    nums[rng.randrange(n)] = 0
    if rng.random() < 0.5:                       # tạo mảng đối xứng để có đáp án > 0
        half = [rng.randint(0, 3) for _ in range(rng.randint(1, 20))]
        nums = half + [0] + half
    vals = {name: f(nums) for name, f in fast.items()}
    assert len(set(vals.values())) == 1, (nums, vals)
print("ngẫu nhiên đúng ràng buộc (các cách nhanh): OK")

vét cạn 4368 mảng nhỏ so với mô phỏng: OK


ngẫu nhiên nhỏ có seed: OK (số ca đáp án > 0: 45 )


ngẫu nhiên đúng ràng buộc (các cách nhanh): OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Mô phỏng ngay trên `nums` (không copy)

Nếu sửa trực tiếp `nums`, mảng đã bị phá sau lần thử đầu tiên, và các lần thử sau chạy trên dữ liệu sai.

In [26]:
def sai_khong_copy(nums):
    count = 0
    n = len(nums)
    for i in range(n):
        if nums[i] != 0:
            continue
        for step in (-1, 1):
            arr = nums                 # SAI: không phải bản sao
            curr, st = i, step
            while 0 <= curr < n:
                if arr[curr] == 0:
                    curr += st
                else:
                    arr[curr] -= 1
                    st = -st
                    curr += st
            if all(v == 0 for v in arr):
                count += 1
    return count

nums = [1, 0, 2, 0, 3]
print(sai_khong_copy(list(nums)), "(sai) vs", count_valid_simulate(nums), "(đúng)")
assert sai_khong_copy(list(nums)) != count_valid_simulate(nums)

0 (sai) vs 2 (đúng)


**Vì sao sai:** `arr = nums` chỉ tạo **tên thứ hai** cho cùng một danh sách. Lần thử thứ nhất làm mảng giảm dần; lần thử sau thấy dữ liệu đã bị đổi. **Sửa:** `arr = list(nums)` (hoặc `nums[:]`) ở **mỗi** lần thử.

### Lỗi 2: Đếm `1` thay vì `2` khi `L == R`

In [27]:
nums = [1, 0, 2, 0, 3]
print(wrong_equal_gives_1(nums), "(sai) vs", Solution().countValidSelections(nums), "(đúng)")
assert wrong_equal_gives_1(nums) == 1 and Solution().countValidSelections(nums) == 2

1 (sai) vs 2 (đúng)


**Vì sao sai:** khi `L == R`, **cả hai hướng** (trái và phải) đều hợp lệ nên là hai lựa chọn khác nhau; mỗi hướng tính một lần ⇒ `+2`. Ví dụ 1 có đáp án `2` đúng vì vị trí `3` được tính hai hướng.

### Lỗi 3: Khi chênh `1` cộng `2` (hoặc chọn nhầm hướng)

In [28]:
nums = [1, 0]          # L = 1, R = 0: chỉ đi trái hợp lệ
print(wrong_diff1_gives_2(nums), "(sai) vs", Solution().countValidSelections(nums), "(đúng)")
assert wrong_diff1_gives_2(nums) == 2 and Solution().countValidSelections(nums) == 1

# Đi PHẢI từ [1, 0]: curr = 1 là 0 -> sang 2 (ra ngoài) khi arr[0] vẫn = 1 -> thất bại
print("đi phải:", simulate_once([1, 0], 1, +1), "| đi trái:", simulate_once([1, 0], 1, -1))
assert simulate_once([1, 0], 1, +1) is False and simulate_once([1, 0], 1, -1) is True

2 (sai) vs 1 (đúng)
đi phải: False | đi trái: True


**Vì sao sai:** khi `L - R == 1` chỉ hướng **trái** (về phía tổng lớn hơn) hợp lệ; đi phải thoát ngay khi bên trái chưa hết. **Sửa:** cộng `1` (không phải `2`) cho trường hợp chênh `1`.

### Lỗi 4: Xét cả vị trí có số dương

In [29]:
nums = [1, 0, 2, 0, 3]
print(wrong_not_only_zero(nums), "(sai) vs", Solution().countValidSelections(nums), "(đúng)")
assert wrong_not_only_zero(nums) != Solution().countValidSelections(nums)

4 (sai) vs 2 (đúng)


**Vì sao sai:** đề chỉ cho phép xuất phát ở vị trí `nums[curr] == 0`. Nếu quên điều kiện `x == 0` ta sẽ cộng cả những vị trí dương (có `l*2 == s` hay chênh `1` nhưng **không được phép chọn**). **Sửa:** chỉ cộng đáp án trong nhánh `x == 0` (`else`/`elif` sau `if x:`).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Mô phỏng một quá trình lặp, đếm số lựa chọn thoả" + ràng buộc nhỏ → viết mô phỏng làm oracle trước**, rồi suy công thức và đối chiếu. Đây là cách an toàn nhất.
- **Quá trình "nảy qua lại" (đảo hướng khi đụng)** thường quy về so sánh **tổng hai phía** — nghĩ đến prefix sum.
- **Tổng tiền tố chạy** (`l += x`) thay cho mảng `pref` khi chỉ cần duyệt một chiều.
- **Đếm theo (vị trí, hướng)**: nhớ rằng hai hướng là hai lựa chọn khác nhau; đối xứng `L == R` → `+2`.
- **Điều kiện chênh `1`**: quy tắc "đi về phía **lớn hơn**" — nên viết rõ hai hàm `can_go_left`, `can_go_right` để tránh nhầm dấu.
- Khi mô phỏng, **luôn copy** dữ liệu gốc và **kiểm tra biên** trước khi đọc `arr[curr]`.

## 15. Biến thể của bài

1. **Số bước** thay vì số lựa chọn: đếm tổng số bước người đi thực hiện khi lựa chọn hợp lệ (mô phỏng, hoặc công thức `~ 2·(L + R)` cộng các bước qua `0`).
2. **Chỉ đếm vị trí** (không phân biệt hướng): mỗi vị trí hợp lệ ít nhất một hướng chỉ tính `1`.
3. **Cho phép bắt đầu ở vị trí khác `0`**: người đi phải xử lý ngay số dương tại chỗ — công thức đổi.
4. **Mảng vòng tròn**: ra khỏi biên thì quay lại đầu kia — quá trình không còn kết thúc theo biên, cần lập luận khác.

In [30]:
def count_valid_positions(nums: List[int]) -> int:
    """Biến thể 2: đếm số VỊ TRÍ xuất phát có ít nhất một hướng hợp lệ."""
    s = sum(nums)
    ans = l = 0
    for x in nums:
        if x:
            l += x
        elif abs(l * 2 - s) <= 1:
            ans += 1
    return ans

print(count_valid_positions([1, 0, 2, 0, 3]))             # 1 (vị trí 3)
print(count_valid_positions([0, 0]))                      # 2
print(count_valid_positions([2, 3, 4, 0, 4, 1, 0]))       # 0

def total_steps(nums, start, step):
    """Biến thể 1: tổng số bước của một lựa chọn (cho dù hợp lệ hay không)."""
    arr = list(nums)
    n = len(arr)
    curr, steps = start, 0
    while 0 <= curr < n:
        steps += 1
        if arr[curr] == 0:
            curr += step
        else:
            arr[curr] -= 1
            step = -step
            curr += step
    return steps

print(total_steps([1, 0, 2, 0, 3], 3, -1))     # 20 (mỗi lần lặp của vòng while là một bước; khớp với trace ở mục 8)

1
2
0
20


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1 | Mô phỏng thật từng (vị trí, hướng) | `O(n² · S)` | `O(n)` | oracle kiểm chứng, `n` nhỏ |
| 2 | Công thức `L`, `R` bằng cắt lát | `O(n²)` | `O(n)` | viết nhanh, dễ hiểu |
| 3 | Tổng tiền tố chạy `l`, `s` | `O(n)` | `O(1)` | chuẩn nộp bài |
| 4 | Mảng `pref` dựng sẵn | `O(n)` | `O(n)` | nhiều truy vấn tổng đoạn |
| 5 | Hàm điều kiện từng hướng | `O(n)` | `O(1)` | cần mở rộng/giải thích rõ ràng |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `simulate_once([0], 0, -1)`, `abs(7 - 9)`, `Solution().countValidSelections([0, 0, 0])`, `Solution().countValidSelections([2, 0])`, `Solution().countValidSelections([2, 0, 1])`.

**Bài 2 (tính tay).** Với `nums = [1, 2, 0, 1, 2]`: tính `s`, rồi `L`, `R` của vị trí `0` duy nhất (`i = 2`), kết luận đáp án, và **mô phỏng tay** từng bước cho hướng trái.

**Bài 3 (điền TODO).** Hoàn thành `count_valid_todo(nums)` theo Cách 3.

In [31]:
def count_valid_todo(nums):
    # TODO 1: s = tổng nums; ans = 0; l = 0
    # TODO 2: duyệt x trong nums
    # TODO 3: nếu x != 0: l += x
    # TODO 4: ngược lại nếu l * 2 == s: ans += 2
    # TODO 5: ngược lại nếu abs(l * 2 - s) == 1: ans += 1
    # TODO 6: trả về ans
    pass

# Tự làm xong rồi mới mở test:
# assert count_valid_todo([1, 0, 2, 0, 3]) == 2
# assert count_valid_todo([2, 3, 4, 0, 4, 1, 0]) == 0

**Bài 4 (tự viết lại).** Viết `count_valid_v2(nums)` bằng **mảng hậu tố** `suf[i]` = tổng `nums[i..n-1]` (không dùng tiền tố): tổng phải của `i` là `suf[i + 1]`, tổng trái là `s - suf[i]`. Kiểm tra bằng cách so với `Solution().countValidSelections` trên 1000 mảng ngẫu nhiên.

**Bài 5 (biến thể / mini challenge).** Viết `list_valid_selections(nums)` trả về **danh sách** các cặp `(vị trí, hướng)` hợp lệ (hướng là `"L"` hoặc `"R"`) bằng **công thức** (không mô phỏng), rồi kiểm chứng độ dài và nội dung bằng mô phỏng `simulate_once`. Thử: `list_valid_selections([1, 0, 2, 0, 3])` phải ra `[(3, "L"), (3, "R")]`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `True` (đi ra ngoài ngay, mảng `[0]` toàn 0), `2`, `6` (`[0,0,0]`: 3 vị trí × 2 hướng), `0` (`[2,0]`: `L = 2`, `R = 0`), `1` (`[2,0,1]`: `L = 2`, `R = 1`, chỉ đi trái). Bài 3: nhớ `elif` thứ hai kiểm tra `abs(l * 2 - s) == 1`.
- **Mức 2 (vừa):** Bài 2: `s = 6`, vị trí `2`: `L = 3`, `R = 3` ⇒ đáp án `2`. Với `L = R = 3` cả hai hướng đều hợp lệ nên đáp án là `2`. Để mô phỏng tay hướng trái, hãy lập bảng `bước — curr — giá trị — hướng — mảng`, rồi dùng `trace_simulate([1, 2, 0, 1, 2], 2, -1)` để kiểm tra từng dòng của bạn.
- **Mức 3 (sâu):** Bài 5: với mỗi `i` có `nums[i] == 0` tính `L`, `R`; thêm `(i, "L")` nếu `L - R in (0, 1)`, thêm `(i, "R")` nếu `R - L in (0, 1)`. Bài 4: `suf = [0] * (n + 1)`, `suf[i] = suf[i + 1] + nums[i]`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [32]:
# Bài 1
assert simulate_once([0], 0, -1) is True
assert abs(7 - 9) == 2
assert Solution().countValidSelections([0, 0, 0]) == 6
assert Solution().countValidSelections([2, 0]) == 0
assert Solution().countValidSelections([2, 0, 1]) == 1        # L = 2, R = 1: chỉ đi trái

# Bài 2: nums = [1, 2, 0, 1, 2]
nums_b2 = [1, 2, 0, 1, 2]
assert sum(nums_b2) == 6
assert sum(nums_b2[:2]) == 3 and sum(nums_b2[3:]) == 3         # L = R = 3
assert Solution().countValidSelections(nums_b2) == 2
assert simulate_once(nums_b2, 2, -1) and simulate_once(nums_b2, 2, +1)

# Bài 3
def count_valid_todo_solution(nums):
    s = sum(nums)
    ans = l = 0
    for x in nums:
        if x != 0:
            l += x
        elif l * 2 == s:
            ans += 2
        elif abs(l * 2 - s) == 1:
            ans += 1
    return ans

assert count_valid_todo_solution([1, 0, 2, 0, 3]) == 2
assert count_valid_todo_solution([2, 3, 4, 0, 4, 1, 0]) == 0

# Bài 4: dùng mảng hậu tố
def count_valid_v2(nums):
    n = len(nums)
    suf = [0] * (n + 1)
    for i in range(n - 1, -1, -1):
        suf[i] = suf[i + 1] + nums[i]
    s = suf[0]
    ans = 0
    for i, x in enumerate(nums):
        if x == 0:
            R = suf[i + 1]
            L = s - suf[i]
            if L == R:
                ans += 2
            elif abs(L - R) == 1:
                ans += 1
    return ans

rng = random.Random(19)
for _ in range(1000):
    n = rng.randint(1, 30)
    nums = [rng.randint(0, 6) for _ in range(n)]
    nums[rng.randrange(n)] = 0
    assert count_valid_v2(nums) == Solution().countValidSelections(nums)

# Bài 5
def list_valid_selections(nums):
    s = sum(nums)
    res = []
    l = 0
    for i, x in enumerate(nums):
        if x == 0:
            r = s - l
            if l - r in (0, 1):
                res.append((i, "L"))
            if r - l in (0, 1):
                res.append((i, "R"))
        l += x
    return res

assert list_valid_selections([1, 0, 2, 0, 3]) == [(3, "L"), (3, "R")]
for _ in range(500):
    n = rng.randint(1, 12)
    nums = [rng.randint(0, 4) for _ in range(n)]
    nums[rng.randrange(n)] = 0
    expected = [(i, d) for i, v in enumerate(nums) if v == 0
                for d, step in (("L", -1), ("R", +1)) if simulate_once(nums, i, step)]
    assert list_valid_selections(nums) == expected, nums
print("Tất cả đáp án đều đúng.")

Tất cả đáp án đều đúng.


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (tổng trái `l` so với tổng toàn mảng `s` tại mỗi số `0`)

```python
class Solution:
    def countValidSelections(self, nums: List[int]) -> int:
        s = sum(nums)
        ans = l = 0
        for x in nums:
            if x:
                l += x
            elif l * 2 == s:
                ans += 2
            elif abs(l * 2 - s) == 1:
                ans += 1
        return ans
```

**Checklist**

- [ ] Chỉ xét xuất phát ở vị trí `nums[i] == 0` (nhánh `else`/`elif` sau `if x:`)?
- [ ] `L == R` → `+2` (cả hai hướng); chênh đúng `1` → `+1` (một hướng)?
- [ ] `l` là tổng **các số dương đã duyệt** (tiền tố), `s - l` là tổng phải (vì `x = 0`)?
- [ ] Nếu mô phỏng: có `list(nums)` để copy, kiểm tra biên `0 <= curr < n` trước khi đọc?
- [ ] Đã thử `[1,0,2,0,3] -> 2`, `[2,3,4,0,4,1,0] -> 0`, `[0] -> 2`, `[1,0] -> 1`?
- [ ] Đã đối chiếu công thức với mô phỏng trên mảng nhỏ?
- [ ] Skeleton đúng `class Solution` / `def countValidSelections(self, nums: List[int]) -> int`?